# Red neuronal funcional: entrenamiento 1981–1997 y prueba 1998–2003, sin estaciones compartidas
Validación cruzada de 5 folds agrupados por estación dentro del entrenamiento. La partición se genera en R y se verifica nuevamente en Python.


In [ ]:
import sys
sys.path.append('funciones')
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import Image, display
from Entrenamiento_red import *


## Comprobaciones de la cabeza MLP
Dimensiones, gradientes y semillas reproducibles para cada q y M, sin entrenamiento.

In [ ]:
datos, W, R, cov_norm, opciones_q, opciones_m = main_cabeza_mlp()

## KAN: resolución de la malla

In [ ]:
# Resolución de la malla para el seno de la función sintética
torch.set_num_threads(1)
x_malla = torch.linspace(-1, 1, 1001, dtype=torch.float64)
x_intermedia = (x_malla[:-1] + x_malla[1:]) / 2
resolucion = []
for intervalos in (4, 8, 12, 16):
    capa = CapaKAN(1, 1, intervalos)
    coeficientes = torch.linalg.lstsq(capa.bases(x_malla), torch.sin(2 * torch.pi * x_malla)).solution
    error = capa.bases(x_intermedia) @ coeficientes - torch.sin(2 * torch.pi * x_intermedia)
    resolucion.append({'intervalos': intervalos, 'RMSE_aproximacion': error.square().mean().sqrt().item()})
tabla_resolucion = pd.DataFrame(resolucion)
print(tabla_resolucion.to_string(index=False))
# Doce intervalos reducen el error de aproximación del seno a cerca de 0,00115

## KAN: comprobación con una función sintética
La KAN debe aproximar la función sintética con RMSE ≤ 0,01 antes de usar datos reales.

In [ ]:
# Comprobación sintética con ajuste, validación y verificación independientes
generador = torch.Generator().manual_seed(2026)
x_sintetica = 4 * torch.rand(2200, 8, generator=generador, dtype=torch.float64) - 2
funcion = torch.sin(torch.pi * x_sintetica[:, 0]) + x_sintetica[:, 1].square()
escalas_salida = torch.tensor([[0.5, 0.75, 1.0, 1.25]], dtype=torch.float64)
y_sintetica = funcion[:, None] * escalas_salida
x_ajuste, x_validacion, x_verificacion = x_sintetica[:1400], x_sintetica[1400:1800], x_sintetica[1800:]
y_ajuste, y_validacion, y_verificacion = y_sintetica[:1400], y_sintetica[1400:1800], y_sintetica[1800:]
resultados_sinteticos = []
for semilla in SEMILLAS:
    establecer_semilla(semilla)
    cabeza = CabezaKAN(neuronas=6, m_fpca=4)
    cabeza.calibrar(x_ajuste)
    optimizador = torch.optim.AdamW(cabeza.parameters(), lr=0.01, weight_decay=0)
    mejor_validacion = float('inf')
    sin_mejora = 0
    for epoca in range(1, 4001):
        cabeza.train()
        optimizador.zero_grad(set_to_none=True)
        perdida = (cabeza(x_ajuste) - y_ajuste).square().mean()
        perdida.backward()
        optimizador.step()
        cabeza.eval()
        with torch.no_grad():
            perdida_validacion = (cabeza(x_validacion) - y_validacion).square().mean().item()
        if perdida_validacion < mejor_validacion - 1e-10:
            mejor_validacion = perdida_validacion
            mejor_estado = {nombre: valor.detach().clone() for nombre, valor in cabeza.state_dict().items()}
            mejor_epoca = epoca
            sin_mejora = 0
        else:
            sin_mejora += 1
        if sin_mejora >= 300:
            break
    cabeza.load_state_dict(mejor_estado)
    with torch.no_grad():
        error_verificacion = cabeza(x_verificacion) - y_verificacion
        rmse_salida = error_verificacion.square().mean(dim=0).sqrt()
        rmse = error_verificacion.square().mean().sqrt().item()
    resultados_sinteticos.append({
        'semilla': semilla, 'mejor_epoca': mejor_epoca,
        'RMSE_validacion': np.sqrt(mejor_validacion), 'RMSE_verificacion': rmse,
        'RMSE_max_salida': rmse_salida.max().item(),
    })
    print(f'KAN sintética, semilla {semilla:04d}, RMSE de verificación = {rmse:.6f}')
    if semilla == SEMILLAS[0]:
        cabeza_sintetica = cabeza
tabla_sintetica = pd.DataFrame(resultados_sinteticos)
print(tabla_sintetica.to_string(index=False))
print(tabla_sintetica[['RMSE_verificacion', 'RMSE_max_salida']].agg(['mean', 'std']).to_string())
assert np.isfinite(tabla_sintetica['RMSE_max_salida']).all()
assert tabla_sintetica['RMSE_max_salida'].le(0.01).all(), 'La KAN supera el RMSE sintético permitido.'
tabla_sintetica.to_csv('Tablas/06_KAN_sintetica.csv', index=False)
# Las tres semillas quedan por debajo de 0,01 en cada una de las cuatro salidas sintéticas

## KAN: activaciones de la primera capa

In [ ]:
# Activaciones de la primera capa en el rango del ajuste sintético
primera_capa = cabeza_sintetica.capas[0]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for entrada, ax in enumerate(axes):
    rejilla = torch.linspace(x_ajuste[:, entrada].min(), x_ajuste[:, entrada].max(), 200, dtype=torch.float64)
    for salida in range(OCULTAS_KAN):
        valores = primera_capa.arista(salida, entrada, rejilla)
        ax.plot(rejilla.numpy(), (valores - valores.mean()).numpy(), label=f'Arista hacia h{salida + 1}')
    ax.set_title(f'x{entrada + 1}')
    ax.legend(fontsize=8)
plt.suptitle('Primera capa KAN · función sintética · semilla 0001')
plt.tight_layout()
fig.savefig('Figuras/06_KAN_activaciones_sinteticas.png', dpi=150)
plt.close(fig)
# En la semilla 0001, las aristas hacia h2 recuperan el seno de x1 y la curvatura de x2

## KAN: ajuste simbólico

In [ ]:
# Ajuste simbólico de la arista de x2 hacia la primera neurona oculta
rejilla = torch.linspace(x_ajuste[:, 1].min(), x_ajuste[:, 1].max(), 200, dtype=torch.float64)
coef_simbolicos, _, r2_simbolico = ajustar_simbolica(primera_capa, 0, 1, rejilla)
tabla_simbolica = pd.DataFrame([{
    'semilla': SEMILLAS[0], 'entrada': 1, 'salida': 0,
    'centro': primera_capa.centro[1].item(), 'escala': primera_capa.escala[1].item(),
    'a0': coef_simbolicos[0].item(), 'a1': coef_simbolicos[1].item(),
    'a2': coef_simbolicos[2].item(), 'a3': coef_simbolicos[3].item(),
    'R2_verificacion': r2_simbolico,
}])
print('Expresión de la arista sintética: a0 + a1*u + a2*u² + a3*u³, u = (x-centro)/escala')
print(tabla_simbolica.to_string(index=False))
# El polinomio aproxima la arista sintética con R² cercano a 0,908 en los puntos intercalados
tabla_simbolica.to_csv('Tablas/06_KAN_simbolica_sintetica.csv', index=False)

## Integración del codificador y la KAN
Parámetros de KAN y MLP igualados (diferencia ≤ 10 %).

In [ ]:
# Partición temporal: entrenamiento 1981-1997; prueba 1998-2003
anios = datos['YEAR'].to_numpy()
particiones = datos['particion'].to_numpy()
idx_entrenamiento = particiones == 'entrenamiento'
idx_prueba = particiones == 'prueba'
assert np.all((anios >= 1981) & (anios <= 2003))
assert np.all(particiones == np.where(anios <= 1997, 'entrenamiento', 'prueba'))
assert idx_entrenamiento.any() and idx_prueba.any()
assert not np.any(idx_entrenamiento & idx_prueba)
assert 'STATION_ID' in datos, 'Falta STATION_ID; vuelve a exportar desde R y actualiza el módulo.'
ids_train = set(np.asarray(datos['STATION_ID'])[idx_entrenamiento].astype(str))
ids_test = set(np.asarray(datos['STATION_ID'])[idx_prueba].astype(str))
assert not (ids_train & ids_test), 'Hay estaciones repetidas entre entrenamiento y prueba.'
assert len(ids_train) >= 5 and len(ids_test) > 0, 'No hay suficientes estaciones para 5 folds y prueba.'
print(f'Entrenamiento (1981-1997): {idx_entrenamiento.sum()} curvas')
print(f'Prueba (1998-2003): {idx_prueba.sum()} curvas')
print('Validación cruzada: 5 folds por estación, solo con entrenamiento')
# Preparar entradas para la KAN
# Solo se utilizan datos de entrenamiento
c_entrenamiento = torch.as_tensor(datos['c'][idx_entrenamiento], dtype=torch.float64)
cov_entrenamiento = torch.as_tensor(cov_norm[idx_entrenamiento], dtype=torch.float64)

# Comprobaciones del codificador y la KAN
comprobaciones = []
parametros_cabezas = []
for q in opciones_q:
    for m in opciones_m:
        for semilla in SEMILLAS:
            establecer_semilla(semilla)
            modelo = ModeloCodificadorCabezaKAN(W, R, q, m)
            with torch.no_grad():
                h_entrenamiento = modelo.codificador(c_entrenamiento, cov_entrenamiento)
                modelo.cabeza.calibrar(h_entrenamiento)
            c_lote = c_entrenamiento[:16]
            cov_lote = cov_entrenamiento[:16]
            coordenadas = modelo(c_lote, cov_lote)
            coordenadas.square().mean().backward()
            gradientes = all(
                p.grad is not None
                and torch.isfinite(p.grad).all().item()
                and p.grad.abs().sum().item() > 0
                for p in modelo.parameters()
            )
            establecer_semilla(semilla)
            replica = ModeloCodificadorCabezaKAN(W, R, q, m)
            with torch.no_grad():
                replica.cabeza.calibrar(replica.codificador(c_entrenamiento, cov_entrenamiento))
                error_replica = (coordenadas - replica(c_lote, cov_lote)).abs().max().item()
            comprobaciones.append({
                'q': q,
                'M': m,
                'semilla': semilla,
                'dimensiones_correctas': coordenadas.shape == (len(c_lote), m),
                'salida_finita': torch.isfinite(coordenadas).all().item(),
                'gradientes_finitos_no_nulos': gradientes,
                'error_replica': error_replica,
            })

        # Comparación de parámetros KAN y MLP
        n_kan = contar_parametros(modelo.cabeza)
        ocultas_mlp = max(1, round((n_kan - m) / (q + m + 3)))
        n_mlp = contar_parametros(CabezaMLP(q, m, ocultas_mlp))
        parametros_cabezas.append({
            'q': q,
            'M': m,
            'ocultas_KAN': OCULTAS_KAN,
            'intervalos': INTERVALOS_KAN,
            'grado': 3,
            'parametros_KAN': n_kan,
            'ocultas_MLP': ocultas_mlp,
            'parametros_MLP': n_mlp,
            'diferencia_pct': 100 * abs(n_mlp - n_kan) / n_kan,
            'parametros_codificador': contar_parametros(modelo.codificador),
        })

# Verificación de resultados
tabla_comprobaciones = pd.DataFrame(comprobaciones)
print(tabla_comprobaciones.to_string(index=False))
assert tabla_comprobaciones[['dimensiones_correctas', 'salida_finita', 'gradientes_finitos_no_nulos']].all().all()
assert tabla_comprobaciones['error_replica'].eq(0).all()
# El gradiente alcanza el codificador y las dos capas KAN
# con inicializaciones reproducibles
tabla_comprobaciones.to_csv('Tablas/06_KAN_comprobaciones.csv', index=False)

tabla_parametros = pd.DataFrame(parametros_cabezas)
print(tabla_parametros.to_string(index=False))
assert tabla_parametros['diferencia_pct'].le(10).all()
tabla_parametros.to_csv('Tablas/06_KAN_parametros.csv', index=False)
print('La KAN devuelve M coordenadas latentes.')

## Entrenamiento de la red funcional
Se comparan las cabezas lineal, MLP y KAN con validación cruzada agrupada por estación. En cada fold las estaciones de ajuste y validación son distintas. Las estaciones de prueba (1998–2003) no aparecen en ninguno de los folds.


In [ ]:
# Búsqueda por validación cruzada de 5 folds y ajuste final
if ESCALA is not None:
    resultados = main_entrenamiento(ESCALA, n_folds=5)


## Resultados
### Parámetros de las cabezas

In [ ]:
pd.read_csv('Tablas/08_red_parametros_cabezas.csv')

### Varianza explicada de la FPCA

In [ ]:
display(pd.read_csv('Tablas/08_red_fpca_varianza.csv'))
display(Image('Figuras/08_red_fpca_sedimentacion.png', width=650))

### Configuración elegida por validación cruzada
Pérdida L² funcional promedio de los cinco folds (tres semillas), calculada solamente con curvas de entrenamiento.

In [ ]:
pd.read_csv('Tablas/08_red_configuracion_elegida.csv')

In [ ]:
pd.read_csv('Tablas/08_red_equivalencia_lineal.csv')

### Valores recortados en 0 al reconstruir las predicciones (entrenamiento, ajuste final)

In [ ]:
pd.read_csv('Tablas/08_red_valores_recortados.csv')

### Pérdida L² de validación cruzada por cabeza, q y λ (M = 5)

In [ ]:
busqueda = pd.read_csv('Tablas/08_red_busqueda_hiperparametros.csv')
print('Folds:', sorted(busqueda.fold.unique()))
display(busqueda.groupby(['cabeza', 'q', 'lambda'])['perdida_validacion'].agg(['mean', 'std']).unstack('lambda').round(3))


### Historial del reajuste final con todo el entrenamiento
Después de elegir los parámetros por validación cruzada, cada cabeza se reajusta con 1981–1997. La cantidad de épocas se fija usando la mediana de las mejores épocas de los folds; los datos de prueba no intervienen en la parada ni en la selección.

In [ ]:
historial = historial_entrenamiento(ESCALA)
grafica_historial(historial, 'Figuras/08_red_curvas_entrenamiento.png')
display(Image('Figuras/08_red_curvas_entrenamiento.png', width=1000))

In [ ]:
# Métricas del reajuste final (solo entrenamiento; tres semillas)
guardada = historial[historial.epoca == historial.mejor_epoca]
display(guardada.groupby('cabeza')[['mejor_epoca', 'L2_entrenamiento', 'RECM_entrenamiento', 'L2_relativo_entrenamiento']].mean().round(3))
